# modular-diffusion: from a caption to a picture at toy size

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.15.0/colab/toy_prompt_to_picture.ipynb)

Two finished pieces joined, nothing trained. THE TOY COLLECTIVE (`colab/toy_collective.ipynb`): small engines that each learned to draw one of nine things in one cell of a 3 x 3 grid, alone, composed into a picture of several things by letting every image token be served by the one engine whose declared address it carries, and a small judge that reads each cell. THE LAYOUT WRITER (`colab/layout_sizes.ipynb`): a language model asked to write one line a thing, `name | color | column | row`, for a text prompt, and a scorer that says whether the written layout satisfies it.

THE CAPTIONS: 600, made by code (`layout.bed_prompts`): the six families of the writer measurement (count, absent, colour, place, relation, combined; 100 each) in their wordings, with the scenes' nine things (disc, ring, crescent, square, triangle, diamond, plus, cross, star) and nine colors; every true layout holds at most one thing a cell. THE CHAIN, per caption: the writer's answer, read by the table-tolerant read and the strict line grammar; the written layout handed on as requests (`layout.bed_requests`: a thing that is none of the nine or an instance without a place is dropped, the first written keeps a cell, a missing or foreign color is drawn from a seeded generator); the collective (the earlier run's engines of seed 0 and of seed 1, fetched from the Hub) draws the requests in 50 steps from the caption's own start noise; the judge reads the nine cells. FOUR READS a caption, each 0 or 1: WRITTEN RIGHT (the written layout satisfies the caption), HANDED RIGHT (the requests, after the rules, satisfy it), DRAWN AS WRITTEN (all nine cells read as the requests say) and PICTURE RIGHT (the layout the judge reads from the picture satisfies the caption). THE WRITERS: a chosen language-model cell (`WRITER_SIZE`, `WRITER_KIND`, `WRITER_EXAMPLES` at the top of the second cell), the 2B base model with six worked examples, a writer of each caption's true layout and a writer of another caption's layout (which must fail).

1. install at the release tag and print the environment;
2. the pins of the sizes needed checked and the models fetched; the earlier run's jobs fetched (the cell stops if one is missing); the controls as finished jobs of the list (the true-layout and the other-caption writers on all 600 captions and their pictures on both seeds; the collective on the earlier run's own lists beside its stored numbers); a smoke of both language-model writers on 8 captions a family through the whole chain with their seconds; the projected minutes; the smoke shipped;
3. the rest of the job list (the two language-model writers and their pictures on both seeds), one job at a time; each finished job folder shipped at once; a progress line a minute; the summary; the machine is given back only when every finished job and the report were shipped.

RESUME: run the cells again. Finished jobs keep their markers under `OUT`; on a new machine finished jobs are fetched back from the Hub, so nothing runs twice.

In [ ]:
# C1 - install at the tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[layout,hub] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.15.0"
import importlib.metadata, subprocess, sys
try:                 # an old torchao beside PEFT stops PEFT from building a LoRA (PEFT wants 0.16 or newer, or none)
    _tao = importlib.metadata.version("torchao")
    if tuple(int(x) for x in _tao.split(".")[:2]) < (0, 16):
        subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
        print("torchao", _tao, "removed: PEFT cannot build a LoRA beside it")
except (importlib.metadata.PackageNotFoundError, ValueError):
    pass
import os, sys, time, json, gc, shutil, threading
import torch, geolip.modular_diffusion as MD
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 15, 0), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
import transformers, huggingface_hub
try:
    import peft
    PEFT = peft.__version__
except ImportError:
    PEFT = "not installed (the plain merge is used)"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| huggingface_hub", huggingface_hub.__version__,
      "| peft", PEFT, "| package", MD.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("card:", props.name, f"| memory {props.total_memory / 2**30:.1f} GiB | cpus: {os.cpu_count()}")
else:
    print("card: NONE (choose a GPU runtime)")
OUT = "/content/toy_prompt_to_picture"       # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

Choose the language-model writer first (`WRITER_SIZE`, `WRITER_KIND`, `WRITER_EXAMPLES`). The base models come from the Hub at their pinned commits (the chosen writer's size and the 2B, which writes beside it with six worked examples); a fine-tune's adapter tag must resolve to its pinned commit, else the cell stops. The earlier run's jobs are fetched from the Hub (the judge, the engines of both seeds, the collective's stored row, the real scenes' row, the wrong-address rows); the cell stops if one is missing, so nothing is ever trained here. Then the controls run as finished jobs of the list: the true-layout writer and the other-caption writer on all 600 captions and their pictures with the engines of each seed (the true-layout writer must read written right and handed right on every caption, and wherever its picture is drawn as written the picture must be right; the other-caption writer's pictures should satisfy their captions at most 0.2 of the time on count, colour, relation and combined), and the collective on the earlier run's own 500 lists of four things beside its stored cell rate (with the judge's stored ceiling and the real scenes at nine things). Then both language-model writers on 8 captions a family through the whole chain on this card with their seconds (the batch starts at 8 and is halved, to 4 then 2, when the card runs out of memory); the job list is projected from those seconds; the smoke folder is shipped.

In [ ]:
# C2 - the writer, the pins, the earlier run, the controls, the smoke, the projection
WRITER_SIZE = "9b"  #@param ["2b", "4b", "9b"]
WRITER_KIND = "lucia"  #@param ["base", "lucia"]
WRITER_EXAMPLES = "six"  #@param ["six", "nine", "none"]
from huggingface_hub import snapshot_download, HfApi
from geolip.modular_diffusion import hub, run, layout
from geolip.modular_diffusion import collective as C
from geolip.modular_diffusion.experiments import layout_write, toy_collective as T
RUNS, SMOKE, FETCHED = f"{OUT}/runs", f"{OUT}/smoke", f"{OUT}/earlier"
REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" = nothing shipped)
PREFIX = "modular_diffusion/conditioning/toy_prompt_to_picture"
EARLIER = "modular_diffusion/conditioning/toy_collective"
NOT_SHIPPED = ("SHIPPED", "FROM_HUB")
DEVICE = "cuda"
EXTRA_WRITE, EXTRA_DRAW = "", ""   # more arguments for every writer cell and every picture job (empty in a real run)
os.makedirs(RUNS, exist_ok=True)
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")
# a size: (the base model, its pinned commit, the fine-tune's adapter, the commit its tag must resolve to)
MODELS = {"2b": ("Qwen/Qwen3.5-2B", "15852e8c16360a2fea060d615a32b45270f8a8fc", "AbstractPhil/qwen3.5-2b-think-symbolic-image", "9366fbbe7846a7452c9dfc59cecfe46eca519c5a"),
          "4b": ("Qwen/Qwen3.5-4B", "851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a", "AbstractPhil/qwen3.5-4b-think-symbolic-image", "4c315a727f69ef2c777d4a58053dfb024304b261"),
          "9b": ("Qwen/Qwen3.5-9B", "c202236235762e1c871ad0ccb60c8ee5ba337b9a", "AbstractPhil/qwen3.5-9b-think-symbolic-image", "ecf2b9a5c3f8eb752fdb5041568c23e0f861129e")}
ADAPTER_TAG = "lucia-v1.2"
EXAMPLES = {"six": 1, "nine": 2, "none": 0}
assert WRITER_SIZE in MODELS and WRITER_KIND in ("base", "lucia") and WRITER_EXAMPLES in EXAMPLES, "the writer's choice"
CHOSEN = layout_write.cell_name(f"{WRITER_KIND}{WRITER_SIZE}", "cell", EXAMPLES[WRITER_EXAMPLES])
SMALL = layout_write.cell_name("base2b", "cell", 1)                  # the 2B base model with six worked examples
assert CHOSEN != SMALL, "the chosen writer is the small writer: choose another cell"
NEEDED = sorted({WRITER_SIZE, "2b"})
BASE = {s: f"{OUT}/base{s}" for s in MODELS}
ADAPTER = {s: f"{OUT}/lucia{s}" for s in MODELS}

# the pins and the fetches (the sizes needed only; an adapter only for the fine-tune)
api = HfApi()
ADAPTER_SHA = {}
for s in NEEDED:
    base, rev, adapter, commit = MODELS[s]
    assert api.model_info(base, revision=rev).sha == rev, f"the {s} base model's commit"
    snapshot_download(repo_id=base, revision=rev, local_dir=BASE[s])
    print(f"{s}: {base} at {rev}")
if WRITER_KIND == "lucia":
    base, rev, adapter, commit = MODELS[WRITER_SIZE]
    ADAPTER_SHA[WRITER_SIZE] = api.model_info(adapter, revision=ADAPTER_TAG).sha
    print(f"{WRITER_SIZE}: {adapter} tag {ADAPTER_TAG} -> {ADAPTER_SHA[WRITER_SIZE]}")
    assert ADAPTER_SHA[WRITER_SIZE] == commit, f"the adapter's tag {ADAPTER_TAG} does not resolve to the pinned commit {commit}: STOP"
    snapshot_download(repo_id=adapter, revision=commit, local_dir=ADAPTER[WRITER_SIZE],
                      allow_patterns=["adapter_config.json", "adapter_model.safetensors"])
    assert all(os.path.exists(f"{ADAPTER[WRITER_SIZE]}/{f}") for f in ("adapter_config.json", "adapter_model.safetensors")), \
        "the adapter's files are not at the root of its repository: look at its file list"

# the earlier run's jobs: the judge, the engines of both seeds, the stored rows; the cell stops if one is missing
EARLIER_JOBS = ("judge_F", "at_F_s0", "ab_F_s0", "at_F_s1", "ab_F_s1", "R1_F_s0", "REAL_F", "wrong_F_s0", "wrong_F_s1")
E = lambda t: f"{FETCHED}/{EARLIER}/{t}"
def earlier_missing():
    # a job is missing when its result is not here, or (the judge, the engines) its weights are not
    miss = [t for t in EARLIER_JOBS if not os.path.exists(f"{E(t)}/result.json")]
    if not os.path.exists(f"{E('judge_F')}/judge.safetensors"):
        miss.append("judge_F weights")
    for s in ("0", "1"):
        found = T.index_engines(E(f"at_F_s{s}")), T.index_engines(E(f"ab_F_s{s}"))
        if len(found[0]) != 9 or len(found[1]) != 1 or not all(
                os.path.exists(f"{p}/{f}") for d in found for p in d.values() for f in C.ENGINE_FILES):
            miss.append(f"the engines of seed {s}")
    return sorted(set(miss))
if earlier_missing():
    snapshot_download(repo_id=REPO, allow_patterns=[f"{EARLIER}/{t}/*" for t in EARLIER_JOBS], local_dir=FETCHED)
MISSING = earlier_missing()
assert not MISSING, f"the earlier run's jobs are not all on the Hub ({', '.join(MISSING)}): STOP, nothing is trained here"
PARTS = {s: f"{E(f'at_F_s{s}')},{E(f'ab_F_s{s}')}" for s in ("0", "1")}
print("the earlier run's jobs fetched:", ", ".join(EARLIER_JOBS))

# the job list: (tag, experiment, arguments, needs); writer cells, then each writer's pictures with each seed's engines
DTYPE = "bf16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "fp16"
BATCH = {"2b": 8, "4b": 8, "9b": 8}
WRITERS = {"chosen": (CHOSEN, WRITER_SIZE, f"writer={BASE[WRITER_SIZE]}" + (f" adapter={ADAPTER[WRITER_SIZE]}" if WRITER_KIND == "lucia" else ""),
                      EXAMPLES[WRITER_EXAMPLES]),
           "small": (SMALL, "2b", f"writer={BASE['2b']}", 1),
           "true": ("true_cell_ex1", None, "writer=fake:truth", 1),
           "next": ("next_cell_ex1", None, "writer=fake:shuffled", 1)}
COMMON = f"device={DEVICE} dtype={DTYPE} max_new_tokens=512 local_only=1 seed=0 threads=4"
def job_list(write_extra="", draw_extra=""):
    jobs = []
    for role, (cell, size, w, ex) in WRITERS.items():
        b = f" batch={BATCH[size]}" if size else ""
        jobs.append((cell, "layout_write", f"variant=write {w} form=cell examples={ex} captions=bed repeat_n=20 {COMMON}{b}{write_extra}", []))
        for s in ("0", "1"):
            jobs.append((f"draw_{cell}_s{s}", "toy_collective",
                         f"variant=draw answers={RUNS}/{cell} parts={PARTS[s]} judge={E('judge_F')} bed=F seed={s} role={role} "
                         f"device={DEVICE} det=on{draw_extra}", [cell]))
    jobs.append(("parity_F_s0", "toy_collective",
                 f"variant=parity parts={PARTS['0']} judge={E('judge_F')} stored={E('R1_F_s0')} real_row={E('REAL_F')} "
                 f"wrong_rows={E('wrong_F_s0')},{E('wrong_F_s1')} bed=F seed=0 device={DEVICE} det=on{draw_extra}", []))
    return jobs
JOBS = job_list(EXTRA_WRITE, EXTRA_DRAW)
BY_TAG = {t: (x, a, n) for t, x, a, n in JOBS}
assert len(BY_TAG) == len(JOBS) == 13, "the job list: five cells of four writers and their eight picture jobs, and the parity"
if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # placed in this process's environment only

def fetch_back(tags):
    # finished jobs come back from the Hub with their files, so nothing runs twice; a job comes back only when every job
    # it reads came back (or finished) too
    if not REPO:
        return
    try:
        got = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX}/{t}/*" for t in tags], local_dir=f"{OUT}/hub_back")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
        return
    for t in tags:
        src, dst = f"{got}/{PREFIX}/{t}", f"{RUNS}/{t}"
        if mark(t, "DONE") or not os.path.exists(f"{src}/result.json"):
            continue
        if BY_TAG[t][0] == "layout_write" and not os.path.exists(f"{src}/outputs.jsonl"):
            continue                                              # its answers are not on the Hub: it is run again
        if not all(mark(n, "DONE") for n in BY_TAG[t][2]):
            continue
        shutil.copytree(src, dst, dirs_exist_ok=True)
        open(f"{dst}/DONE", "w").write("fetched back from the Hub\n")
        open(f"{dst}/FROM_HUB", "w").write("fetched back from the Hub\n")

def ship_job(tag, folder):
    if REPO and hub.ship(folder, REPO, f"{PREFIX}/{tag}", f"toy prompt to picture: {tag}", ignore=NOT_SHIPPED):
        open(f"{folder}/SHIPPED", "w").write("shipped to the Hub\n")
        return True
    return False

SEC = {}
def run_here(tag):
    # a job of the list run in this cell: a finished job of the list (the third cell does not run it again)
    t0 = time.time()
    if not mark(tag, "DONE"):
        shutil.rmtree(f"{RUNS}/{tag}", ignore_errors=True)
        exp, args, _ = BY_TAG[tag]
        run.main(exp, args.split() + [f"out={RUNS}/{tag}"])
        open(f"{RUNS}/{tag}/DONE", "w").write("run in C2\n")
        SEC[tag] = time.time() - t0
    if REPO and not (mark(tag, "SHIPPED") or mark(tag, "FROM_HUB")):
        assert ship_job(tag, f"{RUNS}/{tag}"), f"{tag} was not shipped: check the token and the repository"
    return json.load(open(f"{RUNS}/{tag}/result.json", encoding="utf-8"))

# the controls as finished jobs of the list: the two fake writers on all 600 captions and their pictures on both seeds
CONTROL_TAGS = ["true_cell_ex1", "next_cell_ex1"] + [f"draw_{c}_s{s}" for c in ("true_cell_ex1", "next_cell_ex1") for s in "01"] + ["parity_F_s0"]
fetch_back(CONTROL_TAGS)
R = {t: run_here(t) for t in CONTROL_TAGS}
for s in "01":
    recs = R[f"draw_true_cell_ex1_s{s}"]["records"]
    assert len(recs) == 600 and all(r["W"] == 1 and r["H"] == 1 for r in recs), "control (h): the true layouts read written right and handed right"
    assert all(r["P"] == 1 for r in recs if r["C"] == 1), "the true-layout writer: drawn as written but the picture is not right"
    nxt = R[f"draw_next_cell_ex1_s{s}"]["tables"]["families"]
    i_holds = all(nxt[f]["P"]["mean"] <= 0.2 for f in ("count", "colour", "relation", "combined"))
    tt = R[f"draw_true_cell_ex1_s{s}"]["tables"]
    print(f"seed {s}: the true layouts: picture right {tt['all']['P']['mean']:.3f}, drawn as written {tt['all']['C']['mean']:.3f}; "
          f"the other captions' layouts: picture right on count, colour, relation, combined "
          + ", ".join(f"{nxt[f]['P']['mean']:.3f}" for f in ("count", "colour", "relation", "combined"))
          + f" (at most 0.2: {'HOLDS' if i_holds else 'FAILS'})")
P = R["parity_F_s0"]
print(f"the collective on the earlier run's lists of four things: cell rate {P['parity']['cell_rate']['mean']:.4f} against the stored "
      f"{P['parity']['stored_cell_rate'].get('mean')} (within twice its standard error: {'HOLDS' if P['parity']['holds'] else 'FAILS'}); "
      f"the pictures equal bit for bit: {P['parity']['pictures_equal'] if P['parity']['pictures_equal'] is not None else 'no stored hash'}; "
      f"the judge's stored ceiling and the real scenes at nine things: {'HOLD' if P['judge_control']['holds'] else 'DO NOT HOLD'}")

# the smoke: both language-model writers on 8 captions a family through the whole chain (the engines of seed 0)
shutil.rmtree(SMOKE, ignore_errors=True)
SMOKE_N = 8
SMOKE_SEC, BATCH_TRIALS = {}, []
for role in ("chosen", "small"):
    cell, size, w, ex = WRITERS[role]
    while True:
        t0 = time.time()
        try:
            r = run.main("layout_write", f"variant=write {w} form=cell examples={ex} captions=bed first_n={SMOKE_N} "
                                         f"repeat_n={BATCH[size]} {COMMON} batch={BATCH[size]}{EXTRA_WRITE}".split() + [f"out={SMOKE}/{cell}"])
        except torch.cuda.OutOfMemoryError:
            r = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        BATCH_TRIALS.append({"cell": cell, "batch": BATCH[size], "ran": r is not None, "seconds": time.time() - t0})
        if r is not None:
            break
        print(f"{cell}: out of the card's memory at batch {BATCH[size]}", flush=True)
        assert BATCH[size] > 2, f"{cell}: out of the card's memory at batch {BATCH[size]}: this card cannot run it"
        BATCH[size] //= 2
        shutil.rmtree(f"{SMOKE}/{cell}", ignore_errors=True)
    wsec = time.time() - t0
    t1 = time.time()
    d = run.main("toy_collective", f"variant=draw answers={SMOKE}/{cell} parts={PARTS['0']} judge={E('judge_F')} bed=F seed=0 "
                                   f"role={role} device={DEVICE} det=on{EXTRA_DRAW}".split() + [f"out={SMOKE}/draw_{cell}_s0"])
    SMOKE_SEC[cell] = {"seconds": wsec, "loading": r["seconds_loading"], "answers": r["prompts"] + (r["controls"]["d"].get("asked") or 0),
                       "batch": BATCH[size], "draw_seconds": time.time() - t1, "drawn": d["captions"]}
    tb = d["tables"]["all"]
    print(f"{cell}: {r['prompts']} captions in {wsec:.0f} s (loading {r['seconds_loading']:.0f} s, batch {BATCH[size]}); pictures "
          f"{time.time() - t1:.0f} s; written right {tb['W']['mean']:.2f}, handed right {tb['H']['mean']:.2f}, drawn as written "
          f"{tb['C']['mean']:.2f}, picture right {tb['P']['mean']:.2f}", flush=True)
JOBS = job_list(EXTRA_WRITE, EXTRA_DRAW)            # the batch of a size as settled by the smoke
BY_TAG = {t: (x, a, n) for t, x, a, n in JOBS}

# the projection: a writer cell from its seconds an answer (600 captions and the first 20 asked again in whole batches)
# plus its loading; a picture job from the smoke's seconds a caption (or the measured control jobs)
COST = {}
draw_per = max(v["draw_seconds"] / max(1, v["drawn"]) for v in SMOKE_SEC.values())
for t, x, a, n in JOBS:
    if t in SEC:
        COST[t] = SEC[t]
    elif t in SMOKE_SEC:
        v = SMOKE_SEC[t]
        per = (v["seconds"] - v["loading"]) / max(1, v["answers"])
        COST[t] = v["loading"] + 30 + per * (600 + -(-20 // v["batch"]) * v["batch"])
    elif x == "layout_write":
        COST[t] = 60.0                                   # a fake writer's cell (fetched back, so not timed here)
    else:
        COST[t] = 60 + draw_per * 600
INSTRUMENTS = {"writer": {"size": WRITER_SIZE, "kind": WRITER_KIND, "examples": WRITER_EXAMPLES, "cell": CHOSEN},
               "small_writer": SMALL, "roles": {role: v[0] for role, v in WRITERS.items()}, "earlier_jobs": list(EARLIER_JOBS),
               "smoke_seconds": SMOKE_SEC, "control_job_seconds": SEC, "projected_seconds": COST, "batch": BATCH,
               "batch_trials": BATCH_TRIALS, "adapter_tag": ADAPTER_TAG, "adapter_commits": ADAPTER_SHA,
               "models": {s: MODELS[s] for s in NEEDED}, "dtype": DTYPE,
               "card": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
               "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "peft": PEFT, "package": MD.__version__}}
os.makedirs(SMOKE, exist_ok=True)
json.dump(INSTRUMENTS, open(f"{SMOKE}/instruments.json", "w", encoding="utf-8"), indent=1)
left = [t for t, *_ in JOBS if not mark(t, "DONE")]
print(f"the job list: {len(JOBS)} jobs, {len(JOBS) - len(left)} done in this cell; about {sum(COST[t] for t in left) / 60:.0f} "
      f"projected minutes for the rest on this card with one runner")
if REPO:
    assert hub.ship(SMOKE, REPO, f"{PREFIX}/_smoke", "toy prompt to picture: smoke run", ignore=NOT_SHIPPED), (
        "the smoke folder was not shipped: check the token and the repository before the long run")

One runner works the rest of the job list in order: each language-model writer's cell on the 600 captions, then its pictures with the engines of each seed (each waits for its cell). The control jobs run in the second cell are finished jobs of the list and are not run again. Nothing is trained: if a job of the earlier run is missing here, the cell stops before the roster starts. Each finished job folder is shipped at once to `REPO` (a writer cell with its raw answers, scored rows and controls; a picture job with every caption's record, its tables and its sheet), with the token from the notebook's secret `HF_TOKEN` (placed in this process's environment only; the jobs themselves run without it). At the end the summary, the roster, its log and the instruments are shipped to `_report`. The machine is given back (`RELEASE`) only when the report and every finished job were shipped; if an upload failed, the machine is kept and the cell names the jobs.

In [ ]:
# C3 - the rest of the job list, each finished job shipped, a progress line a minute, the summary, the machine
from geolip.modular_diffusion import jobs
RELEASE = True                     # give the machine back at the end (only when every finished job and the report were shipped)
PROGRESS_S = 60
MISSING = earlier_missing()
assert not MISSING, f"the earlier run's jobs are not all here ({', '.join(MISSING)}): STOP before the roster, nothing is trained here"
tags = [t for t, *_ in JOBS]
roster = f"{OUT}/roster.txt"
cap = lambda t: max(30, int(3 * COST[t] / 60))
with open(roster, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(f"{t} | {cap(t)} | geolip.modular_diffusion.run | {x} {a}" + (f" | needs={','.join(n)}" if n else "")
                       for t, x, a, n in JOBS) + "\nEND\n")
fetch_back(tags)
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")
def after_job(tag, folder, ok):
    if ok and REPO and not ship_job(tag, folder):
        print(f"{tag}: NOT SHIPPED (its folder stays on this machine: {folder})", flush=True)

shipped_or_fetched = lambda t: mark(t, "SHIPPED") or mark(t, "FROM_HUB")
for t in tags:      # a job that finished in an earlier session but whose upload failed: shipped again now
    if REPO and mark(t, "DONE") and not shipped_or_fetched(t):
        after_job(t, f"{RUNS}/{t}", True)

t0 = time.time()
left0 = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; about {left0 / 60:.0f} min to run; one runner")
runner = threading.Thread(target=jobs.run_roster, daemon=True,
                          kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster.log", card_room_mib=None,
                                      ram_free_gb=4.0, max_hours=30.0, claim=True, after_job=after_job, wait_s=5.0,
                                      env={"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4", "HF_HUB_OFFLINE": "1"}))
runner.start()
while runner.is_alive():
    runner.join(PROGRESS_S)
    done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
    left = sum(COST[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    spent = (time.time() - t0) / 60
    rate = (left0 - left) / 60 / spent if left0 > left and spent > 0 else 1.0     # projected minutes done a minute
    print(f"jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about {left / 60 / rate:.0f} min left", flush=True)

summary = T.summarize_drawn(RUNS) + "\n" + layout_write.summarize(RUNS)
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)
for f in ("roster.txt", "roster.log"):
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", f"{REPORT}/{f.replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, f"{PREFIX}/_report", "toy prompt to picture: summary, roster and instruments") if REPO else None
unshipped = [t for t in tags if mark(t, "DONE") and not shipped_or_fetched(t)]
unfinished = [t for t in tags if not mark(t, "DONE")]
print("report shipped:", shipped, "| finished jobs not on the Hub:", ", ".join(unshipped) or "none",
      "| jobs not finished:", ", ".join(unfinished) or "none")
if RELEASE and shipped and not unshipped and not unfinished:
    from google.colab import runtime
    runtime.unassign()
else:
    print("THE MACHINE IS KEPT:", "the report was not shipped" if not shipped else
          f"these finished jobs were not shipped: {', '.join(unshipped)} (run C3 again to retry)" if unshipped else
          f"these jobs did not finish: {', '.join(unfinished)}" if unfinished else "RELEASE is False")